In [7]:
from agents_training_facility import agents
from dotenv import load_dotenv
import os
from openai import AzureOpenAI
import json
from prompts.prompts import  merged_prompt

load_dotenv()
from tools import brain, common, file_handler, programer, utils_handler

In [2]:
damian = agents.CommandCentre('duda', [],'He plans')
maciek = agents.Agent('maciek', [],'Handle all request that need writing and/or execution python code')
ania = agents.Agent('ania', [],'Handle everything connected with files and directory managment')
michal = agents.Agent('michal', [],'general purpose dude, who can do most of the stuff but is not specialized in anything concrete')


In [3]:
system_prompt = '''You are an expert in planning and task delegation, with the ability to select the most suitable agent for any given task. Your job is to carefully evaluate the task at hand and choose the agent whose specialization is best aligned with the task. If none of the agents have a specialization directly related to the task, select a general-purpose agent.
You must return a JSON object with the agent's name and the reason for choosing them.

Example JSON:
json
{ "Zoe": "She can solve math problems" }
Make sure the selection is based on task relevance and the agent's expertise. If no agent is specialized, select a general-purpose agent.'''



In [4]:
agents_list = """"""
for k,v in damian.agent_registry.items():
    if v['field_agent']:
        agents_list += f"Agent name -- {k} -- Agent {k} with mission {v['mission']} and who can use {v['toolset']} \n"

In [ ]:
gpt_name = 'GPT4_O' 
model = 'gpt-4o-ArturG'

api_key = os.getenv(gpt_name + "AZURE_OPENAI_API_KEY")
azure_endpoint = os.getenv(gpt_name + "AZURE_OPENAI_ENDPOINT")
api_version = os.getenv(gpt_name + "OPENAI_API_VERSION")

client= AzureOpenAI(
    api_key=api_key,
    azure_endpoint=azure_endpoint,
    api_version=api_version)  

In [6]:
original_request = 'Tell me a joke, then reverse it and save it into some txt file. Then pain me a Mona Lisa painting'
step = '''step 1. tell a joke. Result 1 Joke has been printed.
step 2. Reverse the joke. Result 2 Joke has been reversed.
step 3. Save the joke to an joke.txt. The joked saved.  
'''

prompt = f'''
USER REQUEST:
{original_request}

EXECUTION HISTORY:
{step}

AVAILABLE AGENTS
{agents_list}


'''

In [7]:

messages = [{"role": "system", "content": merged_prompt}, {"role": "user", "content": prompt}]


request_params = {
    "model": model,
    "messages": messages}

if True:
    request_params['response_format']= {"type":'json_object'}
response = client.chat.completions.create(**request_params)

dict_response = json.loads(response.choices[0].message.content)


In [8]:
dict_response

{'ania': 'Save the reversed joke into a txt file'}

In [2]:
import importlib
import inspect
import os

module = importlib.import_module(file_handler)
functions_list = [func for name, func in inspect.getmembers(module, inspect.isfunction)]
return functions_list

AttributeError: module 'tools.file_handler' has no attribute 'startswith'

In [9]:
# Assuming all .py files are in the tools folder, dynamically load them
def load_modules_from_folder(folder_path):
    modules = []
    for file_name in os.listdir(folder_path):
        if file_name.endswith(".py"):
            module_name = file_name[:-3]  # Strip the '.py' extension
            modules.append(f"tools.{module_name}")
    return modules

In [14]:
from toolbox.toolbox import ToolBox
toolbox = ToolBox()

# Load and store functions from each module in the tools folder
tools_folder = "tools"  # Change this to the correct folder path
modules = load_modules_from_folder(tools_folder)

for module in modules:
    functions = toolbox.load_functions(module)
    toolbox.store(module, functions)

# Print out the tools list with function names and docstrings
print(toolbox.tools())

Module: tools.brain

Module: tools.common
  talk: "
    Answer to user by displaying the text

    Parameters:
    input_text (str): The text to be echoed back.

    Returns:
    str: The conversational response.
    "

Module: tools.file_handler
  check_if_file_exists: "
    Search if specific file exists and learn it path if it does. 

    Parameters:
    filename (str): The name of the file to search for.

    Returns:
    str: The full path to the file if found, or a 'file not found' message if the file does not exist.
    "
  csv_reader: "
    Read a CSV file into a Pandas DataFrame. Additional parameters can be passed to handle
    specific needs like custom delimiters, missing values, or column types.
    
    Parameters:
    filepath (str): The path to the CSV file. Defaults to 'data.csv'.
    **kwargs: Additional keyword arguments to be passed to pandas.read_csv() function.
    
    Returns:
    pd.DataFrame: The content of the CSV file as a Pandas DataFrame. If the file does 

In [15]:
toolbox.tools()

'Module: tools.brain\n\nModule: tools.common\n  talk: "\n    Answer to user by displaying the text\n\n    Parameters:\n    input_text (str): The text to be echoed back.\n\n    Returns:\n    str: The conversational response.\n    "\n\nModule: tools.file_handler\n  check_if_file_exists: "\n    Search if specific file exists and learn it path if it does. \n\n    Parameters:\n    filename (str): The name of the file to search for.\n\n    Returns:\n    str: The full path to the file if found, or a \'file not found\' message if the file does not exist.\n    "\n  csv_reader: "\n    Read a CSV file into a Pandas DataFrame. Additional parameters can be passed to handle\n    specific needs like custom delimiters, missing values, or column types.\n    \n    Parameters:\n    filepath (str): The path to the CSV file. Defaults to \'data.csv\'.\n    **kwargs: Additional keyword arguments to be passed to pandas.read_csv() function.\n    \n    Returns:\n    pd.DataFrame: The content of the CSV file as 

In [4]:
import json
import pandas as pd



In [5]:
df =pd.read_csv('data/my_wardrobe.csv')

In [6]:
df.

,Category,Item
0,Upper Body,T-shirt
1,Upper Body,Long-sleeve shirt
2,Upper Body,Sweater
3,Upper Body,Hoodie
4,Upper Body,Polo shirt
...,...,...
85,Headwear,Flat Cap
86,Headwear,Snapback Cap
87,Headwear,Beret
88,Headwear,Top Hat


In [1]:
# https://dslackw.gitlab.io/colored/tables/colors/

from agents_training_facility import agents, personalities
from dotenv import load_dotenv
import os
import json
from termcolor import colored
from prompts.prompts import  plan_next_step_system, plan_next_step_user, agent_choose_tool_user, synthesis_system, synthesis_user
import traceback
from memory.memory_manager import Memory
import time
load_dotenv()

STEPS_TO_TRACK = 40

### Utils
#-----------------------------------------------------------------------------------
def get_key( diction):
    return list(diction.keys())[0]
def get_value( diction):
    return list(diction.values())[0]




### Agents setup and definition 
#-----------------------------------------------------------------------------------
long_history = Memory(is_structured=False)
step_history = Memory(is_structured=True)
requests_history = Memory(is_structured=False)


manager = agents.CommandCentre('manager', ['brain', 'common'],personalities.brain_desc, personalities.brain_system)
python_developer = agents.Agent('pythondeveloper', ['programer', 'common'],personalities.python_developer_desc, personalities.python_developer_system)
secretary = agents.Agent('secretary', ['file_handler', 'common'],personalities.secretary_desc, personalities.secretary_system)
intern = agents.Agent('intern', ['utils_handler', 'common'],personalities.intern_desc, personalities.intern_system)
api = agents.Agent('api', ['apis'],personalities.communicator_desc, personalities.communicator_system)

agents_dict = {manager.name:manager,
               python_developer.name:python_developer,
               secretary.name:secretary,
               intern.name:intern,
               api.name:api}

agents_list = manager._get_agents_characteristics()
print(agents_list)

Agent name -- manager -- Agent manager || Mission  Brain of the operation that is planning each step -- can ask user for additional data, react to failure and synthesize data || Tools to used: information_synthesis_to_anser_request, talk_to_user 
Agent name -- pythondeveloper -- Agent pythondeveloper || Mission  main focus on writing proper python code and its execution -- suitable for most analysis and modifications || Tools to used: write_python_code, run_python_script, talk_to_user 
Agent name -- secretary -- Agent secretary || Mission  handle everything connected with reading and writing files and directory managment || Tools to used: datetime, check_if_file_exists, search_for_files_with_given_pattern, text_writer, read_text, csv_reader, read_json, talk_to_user 
Agent name -- intern -- Agent intern || Mission  is general purpose agent, who can take most of requests that are not handled by specialists || Tools to used: reverse_string, talk_to_user 
Agent name -- api -- Agent api || 